# BAM-N010 polypropylene nanoparticles
## Data evaluation for homogeneity, characterization, and stability study

<table align="left" style="text-align: left;">
    <tr><td rowspan="4">
        <img align="left" style="padding: 0em 1em 0 0;" src="info/Bottle_PP-Nanoplastics_IMG_3417.jpg" height="100px" width="100px">
    </td><th>Author:</th><th>Andreas F. Thünemann</th></tr>
    <tr><td>Email: </td><td>andreas.thuenemamm@bam.de</td></tr>
    <tr><td>Phone: </td><td>+49 30 8104 1610 </td></tr>
    <tr><td>Address:</td><td>Unter den Eichen 87, 12205 Berlin</td></tr>
</table>

The reference material BAM-N010 consists of an aqueous suspension containing polypropylene nanoparticles.  
The material is provided in glass bottles containing a volume of 10 ml.  
The goal is to certify the hydrodynamic diameter $D_h$.  
Information is additionally given on the polydispersity index PDI and the zeta-potential

## Definitions and Imports

### Loading and configuring required modules

In [ ]:
import os, scipy, glob, sys, re
import pandas as pd
import numpy as np
import scipy
from scipy import stats
# The one-way ANOVA tests the null hypothesis that two or more groups have the same population mean. 
# The test is applied to samples from two or more groups, possibly with differing sizes.
# https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.f_oneway.html
from scipy.stats import f_oneway

from lmfit import Minimizer, Parameters
from lmfit.printfuncs import report_fit

import warnings
# Suppress specific warning
warnings.filterwarnings("ignore", category=SyntaxWarning)

from pathlib import Path
parentDir = str(Path().resolve().parent)
if parentDir not in sys.path:
    sys.path.append(parentDir)

# plotting
import matplotlib
import matplotlib.pyplot as plt
from pyNanoRMcert_tools.Small_Style import configMatplotlib
from pyNanoRMcert_tools.Small_Style import BAMColors
from pyNanoRMcert_tools.Small_Style import BAMMarkerStyles
# rounding
from pyNanoRMcert_tools.round_sig import f_round_sig
from pyNanoRMcert_tools.round_sig import f_round_sig_array

# --- style of plot ---

marker_style_black = dict(color=BAMColors.black, linestyle='', marker='o', fillstyle='none',
                    markersize=10, 
                          markeredgewidth=1, markeredgecolor=BAMColors.black,
                         capsize=5, )

marker_style_red = dict(color=BAMColors.red, linestyle='', marker='s', fillstyle='none',
                    markersize=12, markeredgewidth=1, markeredgecolor=BAMColors.red,
                       capsize=10,)

marker_style_blue = dict(color=BAMColors.blue, linestyle='', marker='o', fillstyle='none',
                    markersize=12, markeredgewidth=1, markeredgecolor=BAMColors.blue,
                        capsize=10,)

marker_style_green = dict(color=BAMColors.green, linestyle='', marker='^', fillstyle='none',
                    markersize=12, markeredgewidth=1, markeredgecolor=BAMColors.green,
                         capsize=10,)

marker_style_yellow = dict(color=BAMColors.yellow, linestyle='', marker='D', fillstyle='none',
                    markersize=12, markeredgewidth=1, markeredgecolor=BAMColors.yellow,
                          capsize=10,)

marker_style_grey = dict(color=BAMColors.black_30, linestyle='', marker='x', fillstyle='none',
                    markersize=12, markeredgewidth=1, markeredgecolor=BAMColors.black_30,
                        capsize=10,)

marker_styles=[marker_style_red, marker_style_blue, marker_style_green, marker_style_yellow, marker_style_grey]

# change font size for axes
plt.rcParams['axes.labelsize'] = 14
plt.rcParams['font.size']= 12

from pyNanoRMcert_tools.utils import store_results, prep_outdir

# Information about software versions
print('Software versions used in this notebook\nPython:', sys.version)
print('scipy: ', scipy.__version__ )
print('numpy: ', np.__version__ )
print('pandas:', pd.__version__ )

## Functions

In [ ]:
def f_values_in_percent(df=None):
    """Calculated std in per cent of mean value"""
    df=df.copy()
    list_fractions=[ df[key]['std']/df[key]['mean']*100 for key in df.keys()]
    df.loc[3]=list_fractions
    df.rename(index={3:'std%'}, inplace=True)
    return df


### Export parameter DataFrame as LaTeX code

In [ ]:
def f_Pandas_df2LaTex(df, file_name="test.txt", caption="Caption",
                      label_columns="\n ", label=r"tab:TestLabel",
                      precision=3):
    """
    Export Pandas data frame to Latex table, derived from:
    https://www.codegrepper.com/code-examples/python/convert+pandas+data+frame+to+latex+file
    *precision*: Sets the number of digits after the decimal point to show.
    """
    
    print('Generated LaTeX code in:', file_name)
    outdir = os.path.dirname(file_name)
    if len(outdir) and not os.path.isdir(outdir):
        os.mkdir(outdir)
    floatfmt = f"{{:.{precision}f}}"
    with open (file_name, "w") as f:
        f.write("\\begin{table}[hbt!]\n")
        f.write("\\centering\n")
        f.write("\caption{" + caption +"}\n")
        f.write("\\begin{tabular}{" + "".join(["c"] * len(df.columns))+"}\n")
        f.write("\\toprule\n")
        f.write(label_columns+"\n")
        f.write("\\midrule\n")
        for _, row in df.iterrows():
            # round the numbers to *precision* decimal places
            f.write(" & ".join([str(round(item, precision) if isinstance(item, float) else item)
                                for item in row]) + "\\\\\n")
        f.write("\\bottomrule\n")
        f.write("\\end{tabular}\n")
        f.write("\\label{" + label + "}\n" )
        f.write("\\end{table}\n")
    return

## Report Ch. 3: Homogeneity study
- performed in January 2022
### Overview of the parameter values on each measurement day
- Production of 3 data frames accociated to day 1, day 2 and day 3 of the homogeneity study

In [ ]:
list_days = ['day1','day2','day3']

l_dfs=[]
for item in list_days:
    # read data
    df = pd.read_excel(os.path.join('data', 'data_homogeneity_2022', 'ZetaSizer_NanoPP.xlsx'),
                                   sheet_name='PP_homogeneity_'+item, skiprows=1, nrows=20)
    # read the date of measurement
    df_date = pd.read_excel(os.path.join('data', 'data_homogeneity_2022', 'ZetaSizer_NanoPP.xlsx'),
                               sheet_name='PP_homogeneity_'+item, skiprows=0, nrows=1, header=None)

    df['Date'] = df_date[1].dt.date[0]
    df['Day']  = "Day "+ item[-1]
    df.rename(columns={'ID': 'Sample ID','diameter (nm)': 'D', 'Zeta Potential (mV)':'zeta'}, inplace=True)

    l_dfs.append(df)
df_homogeneity=pd.concat(l_dfs, ignore_index=True)

print(f"Number of entries: {len(df_homogeneity)}")
df_homogeneity

In [ ]:
df_homogeneity_describe = df_homogeneity[['D', 'PDI',
                                          'zeta',]].describe()#.head(3)
df_homogeneity_describe = df_homogeneity_describe.pipe(f_values_in_percent)
display(df_homogeneity_describe.head(3))

save_df_homogeneity_describe=False
if save_df_homogeneity_describe:
    df_homogeneity_describe.to_excel(os.path.join('data', 'data_homogeneity_2022', 'df_homogeneity_describe.xlsx'))

In [ ]:
df1=df_homogeneity[df_homogeneity['Day']=='Day 1']
df2=df_homogeneity[df_homogeneity['Day']=='Day 2']
df2.reset_index(inplace=True, drop=True)
df3=df_homogeneity[df_homogeneity['Day']=='Day 3']
df3.reset_index(inplace=True, drop=True)
df_day1=df1.copy()
df_day2=df2.copy()
df_day3=df3.copy()

- Plot of diameter, PDI, and zeta-potential

In [ ]:
fig, ax = plt.subplots(3,1, figsize=(7,10))

# Diameter
ax[0].errorbar(list(df1.index), 'D', data=df1, **BAMMarkerStyles.blue, label='Day 1')
ax[0].errorbar(list(df2.index), 'D', data=df2, **BAMMarkerStyles.red, label='Day 2')
ax[0].errorbar(list(df3.index), 'D', data=df3, **BAMMarkerStyles.green, label='Day 3')
ax[0].set(
    xlabel='sample', 
    ylabel=r'$D_h$ (nm)',
    ylim=(100, 250)
)
ax[0].legend(frameon=False)

# PDI
ax[1].errorbar(list(df1.index), 'PDI', data=df1, **BAMMarkerStyles.blue, label='Day 1')
ax[1].errorbar(list(df2.index), 'PDI', data=df2, **BAMMarkerStyles.red, label='Day 2')
ax[1].errorbar(list(df3.index), 'PDI', data=df3, **BAMMarkerStyles.green, label='Day 3')
ax[1].set(
    xlabel='sample', 
    ylabel=r'PDI',
    ylim=(0,0.3)
)
ax[1].legend(frameon=False)

# Zeta potential
ax[2].errorbar(list(df1.index), 'zeta', data=df1, **BAMMarkerStyles.blue, label='Day 1')
ax[2].errorbar(list(df2.index), 'zeta', data=df2, **BAMMarkerStyles.red, label='Day 2')
ax[2].errorbar(list(df3.index), 'zeta', data=df3, **BAMMarkerStyles.green, label='Day 3')
ax[2].set(
    xlabel='sample', 
    ylabel=r'zeta potential (mV)',
    ylim=(-60,-30)
)
ax[2].legend(frameon=False)

fig.tight_layout()#pad=3.0)
plt.show();

### ANOVA
- Nul hypothesis: The data from day 1, 2 and 3 have the same mean
- Sample data for control of the implementation are from

ISO GUIDE 35:2017(E) Reference materials — Guidance for characterization and assessment of homogeneity and stability  
See page 89: Table C.1 — Measurement data of a between-unit homogeneity study
of chromium in soil

#### Implementation

In [ ]:
def f_ISO_Guide_35_ANOVA(df):
    """Perform one-way ANOVA according to ISO Guide 35
    
    The one-way ANOVA tests the null hypothesis that two or more groups have the same population mean. 
    The sample data for control are from
    ISO GUIDE 35:2017(E) Reference materials — Guidance for characterization and assessment of homogeneity and stability
    Page 89, Table C.1 — Measurement data of a between-unit homogeneity study of chromium in soil
    
    see also
    https://en.wikipedia.org/wiki/One-way_analysis_of_variance
    """
    df = df.copy() # a copy of the dataframe provided

    # Step 1: Calculate the mean within each group;  groups are the bottles:
    cols = list(df.keys())
    a = len(df) # number of groups
    n = 3 # where n is the number of data values per group. 
    print('Number of groups a =', a, '\nNumber of data per group n =',n )
    
    # Create a summary table as Excel produces it for "Anova: Single Factor"
    d_sums, d_means = {}, {} # dictionary containing mean of each group
    for i in range(a):
        # sums
        d_sums['Y'+str(i)] =(df.iloc[i][cols[1]]+df.iloc[i][cols[2]]+df.iloc[i][cols[3]])
        # means
        d_means['Y'+str(i)]= (df.iloc[i][cols[1]]+df.iloc[i][cols[2]]+df.iloc[i][cols[3]])/n
        
    # Step 2: Calculate the overall mean: 
    Y = sum([ d_means[key] for key in d_means.keys()])/len(d_means)
    #print('Overall mean =',f_round_sig(Y,5))
    
    # Step 3: Calculate the "between-group" sum of squared differences: 
    SS_between = sum([ n*(d_means[key]-Y)**2 for key in d_means.keys()])
    # The between-group degrees of freedom is one less than the number of groups
    df_between = a-1
    MS_between = SS_between/df_between
    #print('Source of variance between-group: SS =',f_round_sig(SS_between,5), ', df =', df_between, ', MS =',f_round_sig(MS_between,5))

    # Step 4: Calculate the "within-group" sum of squares. 
    # sum of the values within each group
    df['Sum'] = d_sums.values()
    # mean value within each group
    df['Mean'] = d_means.values()
    # sum of squares within each group
    df['SS'] = (df[df.keys()[1]]-df['Mean'])**2+(df[df.keys()[2]]-df['Mean'])**2+(df[df.keys()[3]]-df['Mean'])**2
    # standard deviation of the mean within each group
    df['Variance'] = df['SS']/(n-1)
    df['Sigma'] = np.sqrt(df['Variance'])
    # sum of squares within group
    SS_within = df['SS'].sum()
    # The within-group degrees of freedom is a*(n-1)
    df_within = a*(n-1)
    # Thus the within-group mean square value is 
    MS_within = SS_within/df_within
    #print('Source of variance within-group:  SS =',f_round_sig(SS_within,5), ', df =', df_within, ', MS =',f_round_sig(MS_within,5))
    F_value = MS_between/MS_within
    #print('F_value = ', f_round_sig(F_value,5))

    # find the critical F-Value
    # https://stackoverflow.com/questions/39813470/f-test-with-python-finding-the-critical-value
    import scipy.stats
    # Confidence level of 95% is q=1.-0.05
    F_crit = scipy.stats.f.ppf(q=1-0.05, dfn=df_between, dfd=df_within)
    #print("F_crit  = ", f_round_sig(F_crit,5), "(critical F-Value, corresponding to a confidence level of ",
    #scipy.stats.f.cdf(F_crit, dfn=df_between, dfd=df_within)*100,'%)')

    # check of F value and determination of p-value 
    array4f_oneway = [[df.iloc[i][cols[1]], df.iloc[i][cols[2]],  df.iloc[i][cols[3]]] for i in range(len(df))]
    F_value_2, p_value = f_oneway(*array4f_oneway) 
    #print('p-value =', p_value)
    
    # The between-unit standard deviation (see. formula C.2 at page 90 of ISO GUIDE 35:2017(E))
    if MS_between > MS_within:
        s_bb = np.sqrt((MS_between-MS_within)/n)
    else: s_bb = 0.
    
    # The repeatability standard deviation (see formula C.3 at page 90 of ISO GUIDE 35:2017(E)))
    s_r  = np.sqrt(MS_within)
    
    # Table according to ISO Guide 35:2017(E) Table C.2, page 89
    df_ANOVA_table = pd.DataFrame( 
        {   'Overall Mean':[f_round_sig(Y,5),'',''],
            'Overall Std': [f_round_sig(
                    np.sqrt((SS_between+SS_within)/(df_between+df_within)),3),'',''],
            'Source of variation':['Between bottles','Within bottles','Total'], 
            'Sum of squares':[
                f_round_sig(SS_between,5),
                f_round_sig(SS_within,5),
                f_round_sig((SS_between+SS_within),5)
                ], 
            'Degrees of freedom':[df_between,df_within,df_between+df_within], 
            'Mean square':[ f_round_sig(MS_between,5), f_round_sig(MS_within,5), ''],
            'Standard deviation':[f_round_sig(s_bb,4), f_round_sig(s_r,4),' '],
            'F':[f_round_sig(F_value,5),'',''], 
            'Fcrit':[f_round_sig(F_crit,5),'',''],
            'p-value':[p_value,'',''],
        })
    
    return df, df_ANOVA_table

#### Check against ANOVA function provided by MS excel

##### Test data

In [ ]:
input_list = [
    1, 121.30, 128.74, 119.91,
    2, 120.87, 121.32, 119.24,
    3, 122.44, 122.96, 123.45,
    4,117.60,119.66,118.96,
    5,110.65,112.34,110.29,
    6,117.29,120.79,121.42,
    7,115.27,121.45,117.48,
    8,118.96,123.78,123.29,
    9,118.67,116.67,114.58,
    10,126.24,123.51,126.20,
    11,128.65,122.02,121.93,
    12,126.84,124.72,123.14,
    13,122.61,128.48,126.20,
    14,118.95,123.82,118.11,
    15,118.74,118.23,117.38,
    16,119.74,121.78,121.01,
    17,121.21,123.28,116.38,
    18,129.30,124.10,122.02,
    19,136.81,129.80,128.47,
    20,127.81,117.66,122.90,
]

df_ISO_Guide_35 = pd.DataFrame.from_dict(
    {'Bottle no.':   input_list[0::4],
     'Result no. 1': input_list[1::4],
     'Result no. 2': input_list[2::4],
     'Result no. 3': input_list[3::4]})
#df_ISO_Guide_35.to_excel('Example_ISO_Guide_35.xlsx', index=False)
df_ISO_Guide_35.head(5)

##### Calculate ANOVA test results

In [ ]:
df, df_ANOVA_table = f_ISO_Guide_35_ANOVA(df_ISO_Guide_35)
print("\nANOVA according to ISO Guide 35, example Page 89, Table C.1")
df_ANOVA_table

#### Helper functions for ANOVA

In [ ]:
def f_prepare_for_ANOVA(df=None, par=None):
    """Prepare the data for ANOVA evaluation"""
    df = df.copy()
    
    df1 = df[df['Day']=='Day 1']
    df_4ANOVA = df1[['Sample ID',par]].copy()
    df_4ANOVA.rename(columns={'ID': 'Sample ID', par:par+' Day 1'}, inplace=True)
    
    df2 = df[df['Day']=='Day 2']
    df_4ANOVA[par+' Day 2']=df2[par].values
    
    df3 = df[df['Day']=='Day 3']
    df_4ANOVA[par+' Day 3']=df3[par].values
    return df_4ANOVA

#f_prepare_for_ANOVA(df_homogeneity,par='D')

In [ ]:
def f_plot_homogeneity(df, df_ANOVA=None, par='D', save_ANOVA=False):
    """Plot the results of the homogeneity study"""
    
    fig, ax = plt.subplots(1,1, figsize=[6.4, 4.8])
    
    # --- Diameter ---
    ax.errorbar(list(df.index), 'Mean', 'Sigma', data=df, **marker_style_black)#, label='data')
    
    # one sigma
    x_min = 0; x_max = 21 #25
    x = np.linspace(x_min, x_max,50)
    y_mean = np.ones(len(x))* df_ANOVA['Overall Mean'][0]
    ax.errorbar(x, y_mean, ls='-.', lw=2, color=BAMColors.red)

    y_min = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]-df_ANOVA['Overall Std'][0])
    y_max = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]+df_ANOVA['Overall Std'][0])
    #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.red)
    #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.red)
    
    # two sigma
    y_min = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]-2*df_ANOVA['Overall Std'][0])
    y_max = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]+2*df_ANOVA['Overall Std'][0])
    ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.blue)
    ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.blue)
    
    # mean value +- sigma
    x = [22.]; y = df_ANOVA['Overall Mean'][0]
    uy = df_ANOVA['Overall Std'][0]
    #ax.errorbar(x, y, uy, **marker_style_red)#, label=r'$\left<D\right>\pm 1\sigma$')
    
    # mean value +- 2 sigma
    x = [24.]; 
    y = df_ANOVA['Overall Mean'][0]
    uy = 2*df_ANOVA['Overall Std'][0]
    #ax.errorbar(x, y, uy, **marker_style_blue)#, label=r'$\left<D\right>\pm 2\sigma$')
    #ax.set_xlabel('Sample ID', fontsize=15)
    
    #ax.set_xticks(list(df.index) + [22.,24.])
    ax.set_xticks(list(df.index))
    
    #ax.legend(loc='upper right')
    ax.set_ylim(y-1.5*uy,y+1.5*uy)
    
    # parameter-dependent axis labelling
    if par=='D':
        #list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        #list_xticks=list(df['Sample ID'].values) + ['',r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        list_xticks=list(df['Sample ID'].values)
        ax.set_ylabel(r'$D_\text{h}$ (nm)', fontsize=15)
        #ax.set_ylim(140, 240)
    
    if par=='PDI':
        #list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        #list_xticks=list(df['Sample ID'].values) + [r'',r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        list_xticks=list(df['Sample ID'].values) 
        ax.set_ylabel(r'{}'.format(par), fontsize=15)
        ax.set_ylim(0,.2)
        
    if par=='zeta':
        #list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        #list_xticks=list(df['Sample ID'].values) + [r''.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        list_xticks=list(df['Sample ID'].values)
        ax.set_ylabel(r'Zeta potential (mV)', fontsize=15)
        ax.set_ylim(-60,-30)

    ax.set_xticklabels(list_xticks, rotation = 60)
    ax.set_xlabel("Sample ID")
        
    if save_ANOVA:
        store_results(os.path.join("results", "homogeneity"), f"ANOVA_{par}",
                      (df, df_ANOVA), ("Data", "Table"), saveplot=True)
    plt.show()
    return

#f_plot_homogeneity(df=df_D, df_ANOVA=df_D_ANOVA, par='D', save_ANOVA=True)

#### Calculate ANOVA for each parameter and export results
1. Diameter $D_h$ 
2. PDI 
3. Zeta potential

In [ ]:
for par, punit, out_name, latex_name in (("D", "(nm)", "D", "$D$"),
                                         ("PDI", " ", "PDI", "PDI"),
                                         ("zeta", r"(mV)", "zeta", "zeta potential")):

    
    df, df_ANOVA = f_ISO_Guide_35_ANOVA(f_prepare_for_ANOVA(df_homogeneity, par=par))
    display(df_ANOVA)
    display(df.head(2))

    # column labels
    l1="\\shortstack[l]{Overall\\ Mean \\\ "+punit+"}" # label of column 1
    l2="& \\shortstack[l]{Overall\\ $s$ \\\ "+punit+"}"
    l3="& \shortstack[l]{Source of\\ variation} "
    l4="&  $SS$ "
    l5="&  $df$ "
    l6="& $MS$ "
    l7=" &  $s$ "
    l8="&    F "
    l9="& F$_{crit}$ "
    l10="& $p$-value"
    label_columns=l1+l2+l3+l4+l5+l6+l7+l8+l9+l10+"\\\\"

    file_name = os.path.join("results", "homogeneity", f"ANOVA_{out_name}_latex.tex")
    label    = f"tab:ANOVA_{out_name}"
    caption  = f"ANOVA table for between-bottle homogeneity study of {latex_name}"

    f_Pandas_df2LaTex(df_ANOVA, file_name=file_name, label=label,
                      caption=caption, label_columns=label_columns, precision=3)
    f_plot_homogeneity(df=df, df_ANOVA=df_ANOVA, par=par, save_ANOVA=True)

In [ ]:
df_homogeneity_table=pd.read_excel(os.path.join('results', 'homogeneity', 'ANOVA_D_data.xlsx'))

df_homogeneity_table['No']=df_homogeneity_table.index+1

df_homogeneity_table=df_homogeneity_table[['No','Sample ID', 'D Day 1', 'D Day 2', 'D Day 3', 'Sum',
       'Mean', 'SS', 'Variance', 'Sigma']].rename(columns={'D Day 1': 'Result 1', 'D Day 2': 'Result 2', 'D Day 3': 'Result 3'})

#df_homogeneity_table=df_homogeneity_table.style.format({col: '{:3.1f}' for col in df_homogeneity_table.select_dtypes(include='float').columns})
df_homogeneity_table.keys()

In [ ]:
df=df_homogeneity_table.copy()
#display(df)

# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('Measurement data of the hydrodynamic diameter of the between-unit homogeneity study. Units of Result 1, Result 2, Result 3, Sum, Mean, and Sigma are in nm. Units of SS and Variance are nm2')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size


#for col in df.select_dtypes(include='float'):
for col in df[['Sum','Mean','SS']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.1f}")

df['Variance'] = df['Variance'].map(lambda x: f"{x:02.1f}")
df['Sigma'] = df['Sigma'].map(lambda x: f"{x:01.1f}")

for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        paragraph.add_run(str(value))

save_file=True
if save_file:
    file_name='homogeneity_data.doxc'
    file_path=os.path.join('results', 'homogeneity', file_name)
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df

In [ ]:
## All in one plot
    
def f_plot_homogeneity_all(df_homogenetiy, save_fig=True):
    """Plot all the results of the homogeneity study"""
    
    fig, ax = plt.subplots(3,1, figsize=[6.4, 12])
    
    # --- Diameter ---
    i=-1
    for ax in [ax[0], ax[1], ax[2]]:
        i = i+1
        pars=['D','PDI','zeta']
        par = pars[i]
        print(i, '', par)
        
        df, df_ANOVA = f_ISO_Guide_35_ANOVA(f_prepare_for_ANOVA(df_homogeneity, par=par))
        ax.errorbar(list(df.index), 'Mean', 'Sigma', data=df, **marker_style_black)#, label='data')
        # one sigma
        x_min = 0; x_max = 25
        x = np.linspace(x_min, x_max,50)
        y_min = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]-df_ANOVA['Overall Std'][0])
        y_max = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]+df_ANOVA['Overall Std'][0])
        ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.red)
        ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.red)

        # two sigma
        y_min = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]-2*df_ANOVA['Overall Std'][0])
        y_max = np.ones(len(x))* (df_ANOVA['Overall Mean'][0]+2*df_ANOVA['Overall Std'][0])
        ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.blue)
        ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.blue)

        # mean value +- sigma
        x = [22.]; y = df_ANOVA['Overall Mean'][0]
        uy = df_ANOVA['Overall Std'][0]
        ax.errorbar(x, y, uy, **marker_style_red)#, label=r'$\left<D\right>\pm 1\sigma$')

        # mean value +- 2 sigma
        x = [24.]; y = df_ANOVA['Overall Mean'][0]
        uy = 2*df_ANOVA['Overall Std'][0]
        ax.errorbar(x, y, uy, **marker_style_blue)#, label=r'$\left<D\right>\pm 2\sigma$')
        ax.set_xlabel('Sample ID', fontsize=15)
        ax.set_xticks(list(df.index) + [22.,24.])
        #ax.legend(loc='upper right')
        ax.set_ylim(y-1.5*uy,y+1.5*uy)

        # parameter-dependent axis labelling
        if par=='D':
            list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
            ax.set_ylabel(r'$D$ (nm)', fontsize=15)
            #ax.set_ylim(100, 250)

        if par=='PDI':
            list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
            ax.set_ylabel(r'{}'.format(par), fontsize=15)
            ax.set_ylim(0,.3)

        if par=='zeta':
            list_xticks=list(df['Sample ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
            ax.set_ylabel(r'zeta potential (mV)', fontsize=15)
            ax.set_ylim(-60,-30)
            
        if par=='zeta':
            ax.set_xticklabels(list_xticks, rotation = 60)
        else:
            ax.set_xlabel('')
            ax.set_xticks([])

        
    plt.tight_layout()
    #plt.show()
    if save_fig:
        file_figure=os.path.join("results", "homogeneity_PP_all.png")
        plt.savefig(file_figure, dpi=600)

    return

f_plot_homogeneity_all(df_homogeneity, save_fig=False)

# Short term stability study

In [ ]:
list_temps = ['70°C','40°C','20°C','4°C']

list_df = []
for item in list_temps:
    # read data
    df = pd.read_excel(os.path.join('data', 'data_short_term_stability_2022', "Stability_PP.xlsx"),
                                   sheet_name='PP_stability_'+item, skiprows=1)
    df['temp']= item
    df['T']   = float(item.split('°')[0])+273.15 # Temperature in K
    df['1/T'] = 1/df['T']
    df.rename(columns={'size (nm)': 'D', 'Zeta potential (mV)':'zeta'}, inplace=True)
    list_df.append(df)

df_stability= pd.concat(list_df)
    
header_lines=3
print(f"Number of entries: {len(df_stability)}, showing the first {header_lines}:")
#display(df_stability.head(header_lines))
df = df_stability[df_stability['D'].notnull()] # remove NaN
print(f"Number of entries already measured: {len(df)}")
df2=df.copy() # new data frame to avoid problems with slicing of df

date_start = "2022-01-25"
df2['start']= date_start # start date of experiments
df2['start']= pd.to_datetime(df2['start'])
#df2['times']=(df['date of withdrawal']-df['date of withdrawal'].min()).dt.days.astype('float') 
df2['time']=(df['date of withdrawal']-df2['start'].min()).dt.days.astype('float') 
#display(df2.head(header_lines))
display(df2.head())

df_short_term=df2.copy()

### Start values for time = 0 form the homogeneity study

In [ ]:
df_homogeneity_describe= pd.read_excel(os.path.join('data', 'data_homogeneity_2022', 'df_homogeneity_describe.xlsx'), index_col=0 )
df_homogeneity_describe.head(3)

### Data from ANOVA tables 
- mean values
- std of mean
- $u_{bb}$ contribution of uncertainty between bottle homogeneity
- $u_{rep}$ contribution of uncertainty within bottle homogeneity

In [ ]:
# Diameter Dh
df=pd.read_excel(os.path.join('results', 'homogeneity', 'ANOVA_D_table.xlsx'), index_col=0 )
display(df)
mean=df["Overall Mean"].iloc[0]
u_bb = df["Standard deviation"].iloc[0]
u_rep= df["Standard deviation"].iloc[1]
d_D={'mean': mean, 'u_bb':u_bb, 'u_rep':u_rep}
display(d_D)

# PDI
df=pd.read_excel(os.path.join('results', 'homogeneity', 'ANOVA_PDI_table.xlsx'), index_col=0 )
display(df)
mean=df["Overall Mean"].iloc[0]
u_bb = df["Standard deviation"].iloc[0]
u_rep= df["Standard deviation"].iloc[1]
d_PDI={'mean': mean, 'u_bb':u_bb, 'u_rep':u_rep}
display(d_PDI)

# Zeta potential
df=pd.read_excel(os.path.join('results', 'homogeneity', 'ANOVA_Zeta_table.xlsx'), index_col=0 )
display(df)
mean=df["Overall Mean"].iloc[0]
u_bb = df["Standard deviation"].iloc[0]
u_rep= df["Standard deviation"].iloc[1]
d_Zeta={'mean': mean, 'u_bb':u_bb, 'u_rep':u_rep}
display(d_Zeta)

# summary
df_homogeneity=pd.DataFrame([d_D, d_PDI, d_Zeta]).T
df_homogeneity.rename(columns={0: 'D', 1: 'PDI', 2:'zeta'}, inplace=True)
display(df_homogeneity)

### Start values from homogeneity study

In [ ]:
d ={'date of withdrawal': [date_start, date_start, date_start, date_start],
 'date of measurement': [date_start, date_start, date_start, date_start],
 'ID': ['homogeneity', 'homogeneity','homogeneity','homogeneity'],
 'D': 4 * [df_homogeneity['D']['mean'] ],
 'PDI': 4 * [df_homogeneity['PDI']['mean'] ],
 'zeta': 4 * [df_homogeneity['zeta']['mean'] ],
 'temp': ['4°C', '20°C', '40°C', '70°C'],
 'T': ["", "", "", ""],
 '1/T':["", "", "", ""],
 'start':["", "", "", ""],
 'time':["", "", "", ""]}
df0 = pd.DataFrame.from_dict(d)

df0['start']= date_start # start date of experiments
df0['start']=pd.to_datetime(df0['start'])
df0['T'] = [float(item.split('°')[0])+273.15 for item in list(df0['temp'].values)]
df0['1/T']= 1/df0['T']
df0['time']=(df0['start']-df0['start'].min()).dt.days.astype('float') 
df0
# concatenate start values with values from homogeneity study
#df2=pd.concat([df0,df2])
#df2.head()

- Group the data by time and $T$ and calculated mean and std

In [ ]:
# Replace xx°C with xx because slicing of the multicolum data frame does not work with xx°C as index
df2['temp2']=[ float(item.split('°')[0]) for item in df2['temp'].values ]
df_mean = df2.groupby(['temp','time']).mean()
df_std  = df2.groupby(['temp','time']).std()
df_mean


In [ ]:
temperatures = ['4°C', '20°C',  '40°C', '70°C']

for temperature in temperatures:
    # get the mean values and add the result from homogeneity study for time = 0.
    df_mean2 = pd.concat([df_homogeneity_describe.loc[['mean']], 
               df_mean.loc[(temperature,),][['D','PDI','zeta']]] ).rename(index={'mean':0.})

    df_std2 = pd.concat([df_homogeneity_describe.loc[['std']], 
               df_std.loc[(temperature,),][['D','PDI','zeta']]] ).rename(index={'std':0.})
    df_std2.rename(columns={'D':'uD', 'PDI':'uPDI', 'zeta':'uzeta'}, inplace=True)
    df_std2
    df_mean_std=pd.concat([df_mean2, df_std2], axis=1)
    #display(df_mean_std)
    if temperature =='4°C':
        d_means_stds = {temperature: df_mean_std}
    else:
        d_means_stds[temperature] = df_mean_std
        
for key in d_means_stds.keys(): # convert index to column
    d_means_stds[key].reset_index(inplace=True)
    d_means_stds[key]=d_means_stds[key].rename(columns = {'index':'time'})

display(d_means_stds)

## Data evaluation using non-averaged measurand values

In [ ]:
df_all=pd.concat([df0,df2])
df_all['date of withdrawal']=pd.to_datetime(df_all['date of withdrawal'])
df_all['date of measurement']=pd.to_datetime(df_all['date of measurement'])
df_all

In [ ]:
def residual(pars, x, sigma=None, data=None):
    m = pars['m']
    b = pars['b']
    
    model = b + m*x
    
    if data is None:
        return model
    if sigma is None:
        return model - data
    return (model-data)/sigma

In [ ]:
from scipy.stats.stats import pearsonr

df =df_all

colors = [BAMColors.blue, BAMColors.black, BAMColors.green, BAMColors.red]
list_results=[]
    
for par in ['D','PDI','zeta']:
    fig, ax = plt.subplots(1,4, figsize=(12,5))
    i = -1
    for temp in temperatures:
        i = i+1
        
        #curve fit
        x = df[df['temp']==temp]['time'].values
        y = df[df['temp']==temp][par].values
        
        # calculation correlation coefficient and p-value between x and y
        # https://www.statology.org/correlation-test-in-python/
        r, p = pearsonr(x, y)
        #    Pearson correlation coefficient (r): 0.8076
        #    Two-tailed p-value: 0.0047 , p < 0.05 means statistically significat on a 95% significance level
        print(f"r = {r:3.3f}, p = {p:3.3f}")
        if p< 0.05: print(f"{par}({temp}): correlation  is significant (p-value < 0.05)")
        if p>=0.05: print(f"{par}({temp}): correlation  is NOT significant (p-value > 0.05)")
                        
        pars = Parameters()
        pars.add(name='b', value=y[0], vary=False)
        pars.add(name='m', value= 0.)
        
        #ax[i].errorbar('time', par, yerr=None, data=df[df['temp']==temp], label="data ("+ temp+")", 
        #                   c = colors[i], marker='o', fillstyle='none', ls='',markersize=12, capsize=4,)
        
        fit_without_sigma=True
        if fit_without_sigma:
            
            # fit  data
            fit_lin= Minimizer(residual, pars, fcn_args=(x,), fcn_kws={'data': y})#fcn_kws={'sigma':yerr, 'data': y})
            result = fit_lin.leastsq()
            #print('result.params',result.params)
            
            b= result.params['b'].value # fixed value
            m= result.params['m'].value
            um=result.params['m'].stderr
            
            #fit = residual(result.params, x)
            x2 = np.linspace(0,365,200)
            fit       = residual({'b': b, 'm': m} , x2)
            fit_lower = residual({'b': b, 'm': m-2*um} , x2)
            fit_upper = residual({'b': b, 'm': m+2*um} , x2)
            
            #report_fit(result)
            ax[i].errorbar(x, y, yerr=None, label="data ("+ temp+")", 
                           c = colors[i], marker='o', fillstyle='none', ls='',markersize=12, capsize=4,)
            ax[i].errorbar(x2, fit, ls='-', lw= 2,  color=BAMColors.black,label='fit')
            ax[i].errorbar(x2, fit_lower, ls=':', lw= 2,  color=BAMColors.black,label='')
            ax[i].errorbar(x2, fit_upper, ls=':', lw= 2,  color=BAMColors.black,label='')
            ax[i].fill_between(x2, fit_lower, y2=fit_upper, alpha=0.3, color=BAMColors.black, label='confidence\ninterval' )
            
            ax[i].set(xlabel='Time (days)', ylabel=par)
            if par == 'D': 
                ax[i].set(ylim=(140,200),)
                if i==0: ax[i].set(ylabel='$D$ (nm)')
                else: ax[i].set(ylabel='')
            if par == 'PDI':
                ax[i].set(ylim=(0,.3))
                if i==0: ax[i].set(ylabel='PDI')
                else: ax[i].set(ylabel='')
            if par == 'zeta': 
                ax[i].set(ylim=(-50,-20))
                if i==0: ax[i].set(ylabel='Zeta potential (mV)')
                else: ax[i].set(ylabel='')
            
            #ax[i].legend()
            ax[i].legend(numpoints=1, loc=2, frameon=False)
            plt.tight_layout()
    
            # data frame with fit results
            df_res = pd.DataFrame()
        try: 
            d_res={'par': par, 'temperature': temp}

            for key in result.params.keys():
                d_res[key]=result.params[key].value
                d_res['u'+key]=result.params[key].stderr
                if result.params[key].stderr==None: d_res['u'+key] =0.

            # reduced chi^2
            d_res['redchi']=result.redchi
            # Pearson correlation coefficient (r)
            d_res['r']=r
            # Two-tailed p-value
            d_res['p']=p
            # t-test
            d_res['t']=np.abs(d_res['m'])/d_res['um']
            
            list_results.append(d_res)
        except ValueError:
            print('something went wrong with the *.params')

    save_fig=False
    if save_fig:
        file_name="stability_accelerated_"+par+".png"
        file_figure=os.path.join("results", file_name)
        print("Figure saved as ", file_figure,"\n")
        plt.savefig(file_figure, dpi=600)

df_results = pd.DataFrame(list_results)

In [ ]:
df_results

In [ ]:
# estimation of degradation rate

df_results['T']=[float(item.split('°C')[0])+273.15 for item in df_results['temperature'] ]
df_results['1/T']=1/df_results['T']
df_results['ln_m']= np.log( np.abs(df_results['m']))
df_results['uln_m']= np.abs(df_results['um'])/np.abs(df_results['m'])
df_results

In [ ]:
def f_to_latex(list_index = [0,1,2,3], unit=" nm day$^{-1}$ "):
    """Select values for Latex manuscript"""
    for i in list_index[:2]:
        print(f"${df_results['m'].iloc[i:i+1].values[0]:2.2g} \pm", 
              f"{df_results['um'].iloc[i:i+1].values[0]:2.2g}${unit} at",
              f"{df_results['temperature'].iloc[i:i+1].values[0]},")
    for i in list_index[2:3]:
        print(f"${df_results['m'].iloc[i:i+1].values[0]:2.2g} \pm", 
              f"{df_results['um'].iloc[i:i+1].values[0]:2.2g}${unit} at",
              f"{df_results['temperature'].iloc[i:i+1].values[0]} and")
    for i in list_index[3:4]:
        print(f"${df_results['m'].iloc[i:i+1].values[0]:2.2g} \pm ", 
              f"{df_results['um'].iloc[i:i+1].values[0]:2.2g}${unit} at",
              f"{df_results['temperature'].iloc[i:i+1].values[0]}.")
    
    return

print("The fit results for $D_h$ are $m(T)=$")
f_to_latex(list_index = [0,1,2,3])

print("For PDI the $m(T)=$")
f_to_latex(list_index = [4,5,6,7], unit=" ")

print("Finally, for the $\zeta$\,potentials, the $m(T)=$")
f_to_latex(list_index = [4,5,6,7], unit=" mV ")

# Long-term stability study

In [ ]:
def f_summarize_measurements(dfs, items=['D', 'PDI', 'zeta']):
    """
    Summarizes the mean and standard deviation for specified items across multiple DataFrames.

    Parameters:
    - dfs: list of pandas DataFrames, each containing a 'Date' column and the specified items.
    - items: list of column names to summarize (default: ['D', 'PDI', 'zeta'])

    Returns:
    - A concatenated DataFrame with summary statistics for each input DataFrame.
    """
    summary_list = []

    for df in dfs:
        stats = {'date of measurement': [df['Date'].iloc[0]]}
        for item in items:
            desc = df[item].describe()
            stats[item] = [desc['mean']]
            stats['u' + item] = [desc['std']]
        summary_list.append(pd.DataFrame(stats))

    return pd.concat(summary_list, ignore_index=True)


df_homogeneity_summary=f_summarize_measurements([df_day1, df_day2, df_day3])
df_homogeneity_summary

In [ ]:
# short-term stability study
def f_summarize(df_input):
    df=pd.DataFrame()
    for item in ['D', 'PDI', 'zeta']:
        df_x=df_input.groupby('date of measurement')[item].agg(['mean', 'std'])
        df[item]=df_x['mean']
        df['u'+item]=df_x['std']
    df.reset_index(inplace=True)
    df['date of measurement']=pd.to_datetime(df['date of measurement']).dt.date
    return df

df_short_summary=f_summarize(df_short_term)
df_short_summary

## Study March 2025 
- by "Auszubildende"
- Note: samples were shaken, not vortexed

In [ ]:
import datetime

def f_read_data(file, Day='None', verbose=True, nrows=30):
    """ Read Excel file with data from homogeneity measurements """
    
    df=pd.read_excel(file, nrows=nrows)
    #display(df)
    if verbose:
        display( df.keys())

    dfx=pd.DataFrame()
    dfx['Sample ID']=df['Sample'].str[-3:].astype(str)
    dfx['date']=df['Date']
    #dfx['Measurement ID']=df_d1['Measurement ID'].astype(str)
    dfx['D']=df['Hydrodynamic diameter [nm]']
    dfx['PDI']=df['Polydispersity [%]']/100.
    dfx['zeta']=df['Mean Zeta potential [mV]']
    
    average_data=True
    # average the three measurements taken on each day
    if average_data:
        df_average=pd.DataFrame()
        df_average['D']=dfx.groupby('Sample ID')['D'].mean()
        df_average['PDI']=dfx.groupby('Sample ID')['PDI'].mean()
        df_average['zeta']=dfx.groupby('Sample ID')['zeta'].mean()
        #display(df_average)

    df_average['Date']=pd.to_datetime(df['Date'][0]).date()
    df_average['Day']=Day
    df_average.reset_index(inplace=True)

    if verbose:
        display('df_average.types', df_average.dtypes)

    return dfx#df_average

file=os.path.join('data', 'data_2025_March','2025-03-04_PP_NP_Stabilität.xlsx')
df1=f_read_data(file, Day='Day 1', verbose=False)

file=os.path.join('data', 'data_2025_March','2025-03-05_PP_NP_Stabilität.xlsx')
df2=f_read_data(file, Day='Day 2', verbose=False)

file=os.path.join('data', 'data_2025_March', '2025-03-06_PP_NP_Stabilität.xlsx')
df3=f_read_data(file, Day='Day 3', verbose=False, nrows=27)

df=pd.concat([df1, df2, df3]).reset_index(drop=True)
df.rename(columns={"index": "No."}, inplace=True)
df.head()

### Outlier detection for measurements from March 2025

In [ ]:
def f_outliers(df, Z_score=3, verbose=True):
    """ Find outliers according to Z_score values """


    def f_outlier_removal(df_input, par=None, Z_score=Z_score):
        """Detection of outliers using Z-score"""
        
        df=df_input.copy()
        # Calculate Z-scores
        df['Z-score']= (df[par] - df[par].mean()) / df[par].std()
        #display(df)
        # Identify outliers
        outliers = df[np.abs(df['Z-score']) > Z_score]
        #print(outliers)
        return outliers
    
    pars=['D', 'PDI', 'zeta']
    d_outliers={}
    l_outlierIDs=[]
    for item in pars:    
        d_outliers[item]=f_outlier_removal(df, par=item, Z_score=Z_score)
    
    for item in pars:
        #l_o=list(d_outliers[item]['Sample ID'].values
        l_o=list(d_outliers[item].index)
        l_outlierIDs.extend(l_o)

        if verbose:
            print(f'Outliers indexes for {item}: {l_o}')
            print()
            display(d_outliers[item])

    print(f'number of outliers: {len(l_outlierIDs)} (out of {len(df)})' )
    print('indexes of outliers:', l_outlierIDs)
    
    # remove outliers
    df2=df[~df.index.isin(l_outlierIDs)]
    df2.reset_index(inplace=True, drop=True)
    return df2

#f_outliers(df, Z_score=2, verbose=False)

df=df.pipe(f_outliers, Z_score=2, verbose=False)
df=df.rename(columns={'date': 'date of measurement'})

l_dfs=[]
for item in set(df['date of measurement']):
    df_x=df[df['date of measurement']==item]
    l_dfs.append(f_summarize(df_x))
df_2025_March_summary=pd.concat(l_dfs)
display(df_2025_March_summary)

## Data from Lightsizer 500

In [ ]:
def f_read_DLS(data_dir, files='p*.xlsx', verbose=True):
    ''' Read data from DLS measurements measured with the Anton Paar Litesizer 500 instrument 
    *data_dir* is the directory containing the data files'''

    file_list=glob.glob(os.path.join(data_dir, files))
    print('len(file_list) =', len(file_list),'(number of files)\n')
    
    # dictionary with all data
    d_data={}
    for item in file_list:
        if verbose:
            print (item)
        
        match1 = re.search(r'P\d+', item)
        if match1: 
            match=match1

        match2 = re.search(r'Z\d+', item)
        if match2:
            match=match2
                
        if match:
            file_ID=match.group()
            df=pd.read_excel(item)
            # Generate alphabetical column names
            alphabetical_columns = [chr(i) for i in range(ord('A'), ord('A') + len(df.columns))]
            # Rename the columns
            df.columns = alphabetical_columns
            df.replace('nan', np.nan, inplace=True) # Replace string 'nan' with actual np.nan
            df.fillna(0, inplace=True) # Fill all NaN values with a desired value, e.g., 0
    
            d_data[file_ID]=df
        else:
            print('something went wrong:', item)
    return d_data

#d_data1=f_read_DLS('data', files='p*.xlsx', verbose=False)

In [ ]:
def f_DLS_results(d_data):
    
    ''' Read the relevant data from the measurement files. 
    *d_data* is a dictionary containing the data files
    '''
    
    df_r=pd.DataFrame()
    
    l_measurement_name, l_date, l_sample_ID, l_comment, l_Dh, l_PDI, l_temperature =  [], [], [], [], [], [], []
    
    for item in d_data.keys():
        
        l_measurement_name.append(d_data[item]['B'].iloc[0])
        comment=d_data[item]['B'].iloc[3]
        
        # Replace non-breaking spaces with regular spaces
        comment = comment.replace('\n', ' ')
        
        try:
            sample_ID = re.search(r'ID-Nr\.\s*:\s*(\d+)', comment).group(1)
            l_sample_ID.append(sample_ID)
        except:
            pass
        try:
            sample_ID=re.search(r'ID(\d+)', comment).group(1)
            l_sample_ID.append(sample_ID)
        except:
            pass
    
        l_comment.append(comment)
    
        l_Dh.append(d_data[item]['C'].iloc[5])
        l_PDI.append(d_data[item]['C'].iloc[6]/100)
    
        # find date of measurement
        date_str=d_data[item][d_data[item]['B'].str.contains("Start time", case=False, na=False)]['C'].values[0]
        date_str=str(date_str)
        try:
            date=datetime.datetime.strptime(date_str, "%m/%d/%Y %I:%M:%S %p")
        except:
            date=datetime.datetime.strptime(date_str, '%Y-%m-%d %H:%M:%S')
            
        l_date.append(date.date())
        temperature=d_data[item][d_data[item]['B'].str.contains("Temperature", case=False, na=False)]['C'].values[0]
        l_temperature.append(temperature)
                        
    df_r['measurement_name']=  l_measurement_name
    #df_r['sample_ID'] = l_sample_ID
    df_r['ID'] = l_sample_ID
    df_r['date of measurement']=    l_date
    df_r['temperature'] = l_temperature
    df_r['Dh']=         l_Dh
    df_r['PDI']=        l_PDI
    df_r['comment']=    l_comment
    df_r
    
    return df_r

#f_DLS_results(d_data1)

In [ ]:
d_data1=f_read_DLS(os.path.join('data','data_2025-06-30'), files='p*.xlsx', verbose=False)
df_D1=f_DLS_results(d_data1)
d_data2=f_read_DLS(os.path.join('data','data_2025-07-07'), files='p*.xlsx', verbose=False)
df_D2=f_DLS_results(d_data2)
df_D=pd.concat([df_D1,df_D2])
df_D.rename(columns={'Dh': 'D', 'sample_ID': 'ID'}, inplace=True)

fig, ax = plt.subplots()
ax.errorbar(df_D[0:10].index, df_D['D'][0:10], **marker_style_blue , label='before heat treatment')
ax.errorbar(df_D[0:10].index, df_D['D'][10:20], **marker_style_red, label='after heat treatment 1')
ax.errorbar(df_D[0:10].index, df_D['D'][20:30], **marker_style_green, label='after heat treatment 2')
ax.set(
    xlabel='Sample',
    ylabel='$D_h$ (nm)',
    ylim=(100,250)
)
ax.legend(frameon=False)

df_D.head(2)

### Zeta potential measurements

In [ ]:
def f_zeta_results(d_data):
    ''' Read the relevant data from the measurement files. 
    *d_data* is a dictionary containing the data files
    '''
    
    df_r=pd.DataFrame()
    
    l_measurement_name, l_date, l_sample_ID, l_comment, l_zeta, l_PDI, l_temperature =  [], [], [], [], [], [], []
    
    for item in d_data.keys():
        df=d_data[item]
        
        y=df[df['A'].str.contains("Measurement name", case=False, na=False)]['B'].values[0]
        l_measurement_name.append(y)
        
        y=df[df['B'].str.contains("Start time", case=False, na=False)]['C'].values[0]        
        try:
            l_date.append(y.date())
        except:
            y=datetime.datetime.strptime(y, '%m/%d/%Y %I:%M:%S %p')
            l_date.append(y.date())
    
        comment=df[df['A'].str.contains("Comment", case=False, na=False)]['B'].values[0]
        l_comment.append(comment)
        try:
            start=re.search(r'\d+', comment).start()
            y=comment[start:start+3]
            l_sample_ID.append(y)
        except:
            print('no ID number found')
        y=df[df['B'].str.contains("Mean zeta potential", case=False, na=False)]['C'].values[0]
        l_zeta.append(y)
        
        y=df[df['B'].str.contains("Target temperature", case=False, na=False)]['C'].values[0]
        l_temperature.append(y)
    
    df_r['measurement_name']=  l_measurement_name
    df_r['ID'] = l_sample_ID
    df_r['date of measurement']=    l_date
    df_r['temperature'] = l_temperature
    df_r['zeta']=         l_zeta
    df_r['comment']=    l_comment

    return df_r

In [ ]:
d_data1=f_read_DLS(os.path.join('data','data_2025-06-30'), files='Z*.xlsx', verbose=False)
df_zeta1=f_zeta_results(d_data1)
d_data2=f_read_DLS(os.path.join('data','data_2025-07-07'), files='Z*.xlsx', verbose=False)
df_zeta2=f_zeta_results(d_data2)
df_zeta=pd.concat([df_zeta1,df_zeta2])
df_zeta.reset_index(inplace=True, drop=True)
df_zeta.head(2)

## Heat sterilization does not alter hydrodynamic diameter, PDI, or zeta potential

In [ ]:
# Data before heat treatment, DLS data obtained with Litesizer 500
df_2025_June_before_heat=df_D[0:10][['ID', 'date of measurement','D','PDI' ]].merge(df_zeta[0:10][['ID', 'date of measurement','zeta' ]]).copy()

# Data after heat treatment, DLS data obtained with Litesizer 500 
df_2025_June_after_heat=df_D[10:20][['ID', 'date of measurement','D','PDI' ]].merge(df_zeta[10:20][['ID', 'date of measurement','zeta' ]]).copy()

# Data after heat treatment, DLS data obtained with the instrument from ALV Langen
df_2025_July_after_heat2=df_D[20:30][['ID', 'date of measurement','D','PDI' ]].merge(df_zeta[20:30][['ID', 'date of measurement','zeta' ]]).copy()

In [ ]:
# Before heat treatment
df_2025_June_before_heat_summary=f_summarize(df_2025_June_before_heat)
print('Before heat treatment at 121°C for 90 min')
display(df_2025_June_before_heat_summary)

print('After heat treatment at 121°C for 90 min')
df_2025_June_after_heat_summary=f_summarize(df_2025_June_after_heat)
display(df_2025_June_after_heat_summary)

print('Second measurement series after heat treatment at 121°C for 90 min')
df_2025_July_after_heat2_summary=f_summarize(df_2025_July_after_heat2)
display(df_2025_July_after_heat2_summary)

df_heat_summary=pd.concat([df_2025_June_before_heat_summary, df_2025_June_after_heat_summary, df_2025_July_after_heat2_summary ])
df_heat_summary['time']=l_heat=['before sterilization', '3 h after sterilization', '24 h after sterilization']
df_heat_summary=df_heat_summary[['date of measurement','time', 'D', 'uD', 'PDI', 'uPDI', 'zeta', 'uzeta']].reset_index(drop=True)
display(df_heat_summary)

In [ ]:
def f_plot_heat_summary(df, par='D'):
    fig, ax = plt.subplots()
    ax.errorbar(df_heat_summary.index, par, 'u'+par, data=df, **marker_style_black, label='data')
    
    if par=='D':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'$D_h$ (nm)', fontsize=15)
        ax.set(ylim=(100,250),)

    if par=='PDI':
        #list_xticks=list(df['ID'].values) + [r'$\left<\{}\right>\pm 1\sigma$'.format(par),r'$\left<\{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'{}'.format(par), fontsize=15)
        ax.set(ylim=(0,.3),)

    if par=='zeta':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        ax.set_ylabel('Zeta potential (mV)', fontsize=15)
        ax.set(ylim=(-60,-20),)

        save_results=False
        if save_results:
            store_results(os.path.join("results", "long_term_stability"), f"long_term_{par}",
                          (df, df2), ("Data", "Table"), saveplot=True)
        plt.show()

f_plot_heat_summary(df_heat_summary,'D')
f_plot_heat_summary(df_heat_summary,'PDI')
f_plot_heat_summary(df_heat_summary,'zeta')

In [ ]:
d_ANOVA_heat={}
for par in ['D', 'PDI', 'zeta']:
    df_res1=df_2025_June_before_heat[['ID', par]].rename(columns={par: 'before heat'})
    df_res2=df_2025_June_after_heat[['ID', par]].rename(columns={par: 'after heat 1'})
    df_res3=df_2025_July_after_heat2[['ID', par]].rename(columns={par: 'after heat 2'})
    df_merged=df_res1.merge(df_res2).merge(df_res3)
    d_ANOVA_heat[par]=df_merged.copy()
d_ANOVA_heat.keys()

In [ ]:
df=d_ANOVA_heat['D'].copy()
df=df.rename(columns= {'before heat': 'result 1', 'after heat 1': 'result 2', 'after heat 2': 'result 3'})
#display(df)

# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('Hydrodynamic diameter of 10 samples of BAM-N010 before heat sterilization (result 1), 3 h after sterilization (result 2), and 24 h after heat sterilization (result 3). Values are in nm.')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size


for col in df.select_dtypes(include='float'):
    #for col in df[['D','uD']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.1f}")

#for col in df[['PDI','uPDI']].select_dtypes(include='float'):
#    df[col] = df[col].map(lambda x: f"{x:03.3f}")

#for col in df[['zeta','uzeta']].select_dtypes(include='float'):
#    df[col] = df[col].map(lambda x: f"{x:02.1f}")


#df['Variance'] = df['Variance'].map(lambda x: f"{x:02.1f}")
#df['Sigma'] = df['Sigma'].map(lambda x: f"{x:01.1f}")

for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        #paragraph.add_run(str(value))
        run=paragraph.add_run(str(value))
        run.font.name = 'BAM Klavika Light'
        run.font.size = Pt(10)  # Optional: set font size

save_file=False
if save_file:
    file_name='sterilization_D.docx'
    file_path=os.path.join('results', 'sterilization', file_name)
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df

In [ ]:
df, df_ANOVA= f_ISO_Guide_35_ANOVA(d_ANOVA_heat['D'])
df

In [ ]:
par='D'
df, df_ANOVA= f_ISO_Guide_35_ANOVA(d_ANOVA_heat[par])
#display(df)

# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('Hydrodynamic diameter of 10 samples of BAM-N010 before heat sterilization (before heat), 3 h after sterilization (after heat 1), and 24 h after heat sterilization (after heat 2). Units of before heat, after heat 1, after heat 2, Sum, Mean, and Sigma are in nm. Units of SS and Variance are nm2.')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size


#for col in df.select_dtypes(include='float'):

for col in df[['before heat','after heat 1', 'after heat 2','Sum','Mean','SS']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.1f}")

df['Variance'] = df['Variance'].map(lambda x: f"{x:02.1f}")
df['Sigma'] = df['Sigma'].map(lambda x: f"{x:01.1f}")

for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        paragraph.add_run(str(value))

save_file=True
if save_file:
    file_name='sterilization_data_'+par+'.docx'
    file_path=os.path.join('results', 'sterilization', file_name)
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df

In [ ]:
par='PDI'
df, df_ANOVA= f_ISO_Guide_35_ANOVA(d_ANOVA_heat[par])
#display(df)

# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('PDI of 10 samples of BAM-N010 before heat sterilization (before heat), 3 h after sterilization (after heat 1), and 24 h after heat sterilization (after heat 2). Units of before heat, after heat 1, after heat 2, Sum, Mean, and Sigma are in nm. Units of SS and Variance are nm2.')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size


#for col in df.select_dtypes(include='float'):

for col in df[['before heat','after heat 1', 'after heat 2','Sum','Mean']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.3f}")

df['SS'] = df['SS'].map(lambda x: f"{x:01.4f}")
df['Variance'] = df['Variance'].map(lambda x: f"{x:01.4f}")
df['Sigma'] = df['Sigma'].map(lambda x: f"{x:01.4f}")

for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        paragraph.add_run(str(value))

save_file=True
if save_file:
    file_name='sterilization_data_'+par+'.docx'
    file_path=os.path.join('results', 'sterilization', file_name)
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df

In [ ]:
par='zeta'
df, df_ANOVA= f_ISO_Guide_35_ANOVA(d_ANOVA_heat[par])
#display(df)

# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('Zeta potential of 10 samples of BAM-N010 before heat sterilization (before heat), 3 h after sterilization (after heat 1), and 24 h after heat sterilization (after heat 2). Units of before heat, after heat 1, after heat 2, Sum, Mean, and Sigma are in mV. Units of SS and Variance are mV2.')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size


#for col in df.select_dtypes(include='float'):

for col in df[['before heat','after heat 1', 'after heat 2','Sum','Mean','SS']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.1f}")

df['Variance'] = df['Variance'].map(lambda x: f"{x:02.1f}")
df['Sigma'] = df['Sigma'].map(lambda x: f"{x:01.1f}")

for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        paragraph.add_run(str(value))

save_file=True
if save_file:
    file_name='sterilization_data_'+par+'.docx'
    file_path=os.path.join('results', 'sterilization', file_name)
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df

In [ ]:
save_file=True

for par in ['D', 'PDI', 'zeta']:
    df, df_ANOVA_table= f_ISO_Guide_35_ANOVA(d_ANOVA_heat[par])
    print(f'\n parameter = {par}')
    display(df_ANOVA_table)
    
    if save_file:
        file_name='sterilization_ANOVA_'+par+'.xlsx'
        file_path=os.path.join('results', 'sterilization', file_name)
        print(f'Table saved as {file_path}')
        df_ANOVA_table.to_excel(file_path, index=False)

In [ ]:
fig, ax = plt.subplots()
ax.errorbar(df_2025_June_before_heat.index, df_2025_June_before_heat['D'], **marker_style_blue , label='before heat treatment')
ax.errorbar(df_2025_June_after_heat.index, df_2025_June_after_heat['D'], **marker_style_red, label='after heat treatment 1')
ax.errorbar(df_2025_July_after_heat2.index, df_2025_July_after_heat2['D'], **marker_style_green, label='after heat treatment 2')
ax.set(
    xlabel='Sample',
    ylabel='$D_h$ (nm)',
    ylim=(100,250)
)
ax.legend(frameon=False)

par='D'
df_heat_summary

### Export as Word file

In [ ]:
df=df_heat_summary.copy()
#display(df)

# save it as a Word table
#%pip install pandas python-docx
from docx import Document
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.shared import Pt, RGBColor

doc = Document()
title=doc.add_heading(level=1)
run= title.add_run('Mean values and standard uncertainty of hydrodynamic diameter, PDI, and zeta potential of 10 samples of BAM-N010 before, 3 h after, and 24 h after heat sterilization.')
title.alignment = WD_ALIGN_PARAGRAPH.JUSTIFY #CENTER

run.font.name='BAM Klavika Light'
run.font.size = Pt(10) # Optional: adjust size
run.font.color.rgb = RGBColor(0,0,0) # Black

table = doc.add_table(rows=1, cols=len(df.columns))
table.style = 'Table Grid'

hdr_cells = table.rows[0].cells
for i, col_name in enumerate(df.columns):
    paragraph = hdr_cells[i].paragraphs[0]
    paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
    run = paragraph.add_run(col_name)
    run.font.name = 'BAM Klavika Light'#'Arial'
    run.font.size = Pt(10)  # Optional: set font size


#for col in df.select_dtypes(include='float'):
for col in df[['D','uD']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.1f}")

for col in df[['PDI','uPDI']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:03.3f}")

for col in df[['zeta','uzeta']].select_dtypes(include='float'):
    df[col] = df[col].map(lambda x: f"{x:02.1f}")


#df['Variance'] = df['Variance'].map(lambda x: f"{x:02.1f}")
#df['Sigma'] = df['Sigma'].map(lambda x: f"{x:01.1f}")

for _, row in df.iterrows():
    row_cells = table.add_row().cells
    for i, value in enumerate(row):
        paragraph = row_cells[i].paragraphs[0]
        paragraph.alignment = WD_ALIGN_PARAGRAPH.RIGHT
        #paragraph.add_run(str(value))
        run=paragraph.add_run(str(value))
        run.font.name = 'BAM Klavika Light'
        run.font.size = Pt(10)  # Optional: set font size

save_file=True
if save_file:
    file_name='Sterilization_summary.docx'
    file_path=os.path.join('results', 'sterilization', file_name)
    print(f'Table saved as {file_path}')
    doc.save(file_path)

df

## Data 2023 August; 2024 January

In [ ]:
d_data1=f_read_DLS(os.path.join('data','data_2023'), files='p*.xlsx', verbose=False)
df_D1=f_DLS_results(d_data1)
df_D1.rename(columns={'Dh': 'D'}, inplace=True)
#display(df_D1)
d_data1=f_read_DLS(os.path.join('data','data_2023'), files='Z*.xlsx', verbose=False)
df_zeta1=f_zeta_results(d_data1)
#display(df_zeta1)

In [ ]:
date=datetime.date(2023, 8, 16)
df_2023_08_16=df_D1[df_D1['date of measurement']==date][['date of measurement', 'D', 'PDI']]
df_2023_08_16['zeta']=df_zeta1[df_zeta1['date of measurement']==date]['zeta']
#display(df_2023_08_16)
df_2023_August_summary=f_summarize(df_2023_08_16)
df_2023_August_summary

In [ ]:
date=datetime.date(2024, 1, 5)
df_2024_01_05=df_D1[df_D1['date of measurement']==date][['date of measurement', 'D', 'PDI']]
df_2024_01_05['zeta']=df_zeta1[df_zeta1['date of measurement']==date]['zeta']
#display(df_2024_01_05)
df_2024_January_05_summary=f_summarize(df_2024_01_05)
df_2024_January_05_summary

In [ ]:
date=datetime.date(2024, 1, 8)
df_2024_01_08=df_D1[df_D1['date of measurement']==date][['date of measurement', 'D', 'PDI']]
df_2024_01_08['zeta']=df_zeta1[df_zeta1['date of measurement']==date]['zeta']
#display(df_2024_01_08)
df_2024_January_08_summary=f_summarize(df_2024_01_08)
df_2024_January_08_summary

## Plot of long-term study

In [ ]:
df_long_term= pd.concat([df_homogeneity_summary, df_short_summary, 
                         df_2023_August_summary, df_2024_January_05_summary, df_2024_January_08_summary,
                         df_2025_March_summary, df_2025_June_before_heat_summary, df_2025_June_after_heat_summary, df_2025_July_after_heat2_summary], ignore_index=True)
df_long_term['date of production']= pd.to_datetime('2022-01-20')
df_long_term['time']=(pd.to_datetime(df_long_term['date of measurement'])-df_long_term['date of production']).dt.days
df_long_term

The stability was assessed according to ISO Guide 35, using a linear regression 
\begin{equation}
y_{lts} = b_0 + b_1 t,
\end{equation}
where $b_0$ is the intercept and $b_1$ the slope. Their estimated standard errors are $s(b_0)$ and $s(b_1)$, respectively.

Next, a test for statistically significant slopes different from zero was conducted.
The $t$-test statistics for slopes $b_1$ significance were calculated using 
\begin{equation}
t_b= \frac{\left| b_1 \right|}{s(b_1)},
\end{equation}
and compared with the critical values at the 95% level of confidence.
Results of linear regression are presented in Figure 5 and values of $t$-statistics in Table xxx. 
For all $D_h$, the slope is not significantly different from zero, and thus, no significant instability is detected.
By contrast, for PDI and zeta potential, the slopes are significantly different from zero, and thus, a significant instability is detected.


In [ ]:
from scipy.stats import linregress

def f_long_term_model_fit_simple(df=None, par=None, save_results=False, verbose=True):
    """ Reference material long-term stability model fit according to ISO GUIDE 35 B.3.2 """

    #x      = df['Time']/365. # time in years
    x      = df['time']/30. # time in month
    x_mean = x.mean()
    sx2    = np.sum((x-x_mean)**2)

    y       = df[par]
    y_mean  = y.mean()
    y_std   = df.describe()[par]['std']
    
    print(f'\nParameter under investitation is {par}')
    n       = len(y)
    print(f'number of data points n = {n}')
    print(f'mean of  {par}  = {y_mean:.2e} ± {y_std:.2e}' )

    # cacluation of slope b1 according to ISO GUIDE 35:2017(E), 
    # slope according to equation (B.14) at page 82 
    b1 = np.sum( (x-x_mean)*(y-y_mean))/sx2
    # intercept according to equation (B.15)
    b0 = y_mean-b1*x_mean
    
    # standard deviations of b0 and b1
    # eq. (B.17) page 82
    s2 = np.sum( (y-b0-b1*x)**2/(n-2))
    s = np.sqrt(s2)
    # s(b1) according to eq. (B.16)
    s_b1= s /np.sqrt(sx2)
    # standard deviation s(b0)
    s_b0= s_b1*np.sqrt((np.sum(x**2))/n)
    
    print('\nLinear regression according to clause B.3 of ISO GUIDE 35:2017(E)')
    print(f'b0 = {b0:.2e} ± {s_b0:.2e} (intercept)')
    print(f'b1 = {b1:.2e} ± {s_b1:.2e} (slope)\n')

    # B.3.3 Inspection and check of assumptions
    # calculation of residuals
    df_fit=pd.DataFrame()
    #n_years = 5 # number of additional years as x-axis
    #df_fit['x_fit'] = np.linspace(x.min(), x.max()+n_years,1000)
    # total number of years on the x-axis
    
    n_time = 60.# maximum time
    x_min = 0 # 20./365. # for logscale
    df_fit['x_fit'] = np.linspace(x_min, n_time,1000)
    df_fit['y_fit'] = b0+b1*df_fit['x_fit']

    # B.3.4 Testing for statistically significant change
    # equation (B.19) at page 82
    t_b1= np.abs(b1)/s_b1
    print('\nTesting for statistical significant change (ISO GUIDE 35: 2017(E) B.3.4)')
    print('t-test statistics for slope significantly different from zero')
    print(f't_b1 = {t_b1:.2e}')

    # ... and comparing this with the two-tailed critical value of Student’s t for n-2 degrees of freedom at the 95 % level 
    # of confidence. If the calculated test statistic tb1 exceeds the critical value, 
    # the slope is considered to be significantly different from zero at the 95 % level of confidence.
    
    # Searching the student-t distribution table for values using Python
    # https://stackoverflow.com/questions/66872064/searching-the-student-t-distribution-table-for-values-using-python
    # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.t.html
    from scipy.stats import t

    alpha = 0.05  # significance level = 95% 

    degrees_of_freedom = n-2  # degrees of freedom
    t95 = t.ppf(1 - alpha/2, degrees_of_freedom)
    print(f't95 = {t95:.2e} (degrees of freedom = {degrees_of_freedom}, level of significance = 95%)')
    # Consistency check from NIST reference data table on critical values of the Student's t distribution
    # https://www.itl.nist.gov/div898/handbook/eda/section3/eda3672.htm
    # For a two-sided test, we compute 1 - α/2, or 1 - 0.05/2 = 0.975 when α = 0.05
    # for degrees of freedom = 101-2=99 and a significance level of 95%  provides
    # t95 = 1.984

    significance = (t_b1 > t95)
    print('significance: ', significance)
    if significance: 
        print(f't_b1 ({t_b1:.2e}) > t95 ({t95:.2e}) (Interpretation: slope is different from 0)\n')

        # Confidence interval for the regression line (B.3.5 at page 83 of ISO GUIDE 35:2017(E))
        y_fit_delta = t95*s*np.sqrt(1./n + ((df_fit['x_fit']-x.mean())**2/sx2))

        df_fit['y_fit_min']   = df_fit['y_fit'] - y_fit_delta
        df_fit['y_fit_max']   = df_fit['y_fit'] + y_fit_delta
    else:
        print(f't_b1 {t_b1:.2e} < t95 {t95:.2e} (slope is 0)')
        
        # Confidence interval for the regression line (B.3.5 at page 83 of ISO GUIDE 35:2017(E))
        y_fit_delta = t95*s*np.sqrt(1./n + ((df_fit['x_fit']-x.mean())**2/sx2))

        df_fit['y_fit_min']   = df_fit['y_fit'] - y_fit_delta
        df_fit['y_fit_max']   = df_fit['y_fit'] + y_fit_delta

    if verbose:
        # fit linear model
        # https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.linregress.html

        #p-value: float
        #Two-sided p-value for a hypothesis test whose null hypothesis is
        #that the slope is zero, using the Wald Test with a t-distribution of
        #the test statistic.
        model=linregress(x, y)
        #print('linregress(x, y)',linregress(x, y))

        print('\nCalcuation with scipy.stats.linregress')
        print(f'slope    = {model.slope:.2e} ± {model.stderr:.2e}' )
        print(f'intercept= {model.intercept:.2e} ± {model.intercept_stderr:.2e}')
        print(f'rvalue   = {model.rvalue:.2e}')
        print(f'pvalue   = {model.pvalue:.2e}, (Test hypothesis that slope is zero)')

        def f_linear(x, slope, intercept):
            return slope*np.array(x)+intercept

        # Calculate 95% confidence interval on slope and intercept:
        # Two-sided inverse Student's t-distribution
        # p - probability, df - degrees of freedom

        from scipy.stats import t
        tinv = lambda p, df: abs(t.ppf(p/2, df))
        ts = tinv(0.05, len(x)-2)
        print(f"slope     (95%): {model.slope:.2e} ± {ts*model.stderr:.2e}")
        print(f"intercept (95%): {model.intercept:.2e}"f" ± {ts*model.stderr:.2e}")

    # --- plot ---
    fig, ax = plt.subplots(1,1, figsize=[6.4, 4.8])
    
    # Plot with error bars if uncertainties are given.
    if 'u2'+par in df.keys():
        uy=df['u'+par].values
        ax.errorbar(x, y, uy, **marker_style_black, label='data')
    else:
        ax.errorbar(x, y, **marker_style_black, label='data')
    
    
    ax.errorbar(df_fit['x_fit'], y_mean*np.ones(len(df_fit)), color=BAMColors.red, ls=':',lw=2) 

    # --- store the parameters in a Pandas data frame
    df_pars=pd.DataFrame({
        'par' : [par],
        'y_m' : [y_mean],
        's'   : [s],
        'b0'  : [b0],
        's_b0': [s_b0],
        'b1'  : [b1], 
        's_b1': [s_b1],
        't_b1': [t_b1],
        't95' : [t95],
    })
    display(df_pars)

    # -------------------------------------    

    if verbose==True:
        ax.errorbar(x, f_linear(x,model.slope,model.intercept), color=BAMColors.green, label='scipy')
    if 1>0: # plot lines in any case #significance: # plot if slope is significantly larger than 0
        ax.errorbar('x_fit', 'y_fit', data=df_fit, lw=2, color=BAMColors.blue, label='regression')
        #ax.legend()

    plot_verbose=True
    if plot_verbose:
        # one sigma
        x_min = x.min(); 
        x_max = x.max()+1#+365.
        x = np.linspace(x_min, x_max,50)
        y_min = np.ones(len(x))* (y_mean-y_std)
        y_max = np.ones(len(x))* (y_mean+y_std)
        #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.red)
        #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.red)

        # two sigma
        y_min = np.ones(len(x))* (y_mean-2*y_std)
        y_max = np.ones(len(x))* (y_mean+2*y_std)
        #ax.errorbar(x, y_min, ls=':', lw=2, color=BAMColors.blue)
        #ax.errorbar(x, y_max, ls=':', lw=2, color=BAMColors.blue)

        # mean value +- sigma
        #x = [x_max-800/365]
        x = [n_time-1]
        #ax.errorbar(x, y_mean, y_std, **marker_style_red)#, label=r'$\left<D\right>\pm 1\sigma$')
        # mean value +- 2 sigma
        #x = [x_max-200/365]
        x = [n_time]
        #ax.errorbar(x, y_mean, 2*y_std, **marker_style_blue)#, label=r'$\left<D\right>\pm 2\sigma$')

    ax.set_xlabel('Time (months)')
    #ax.set_xscale('log')
    #ax.legend()

    # parameter-dependent axis labelling
    uy = y.max()

    if par=='D':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par),r'$\left<{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'$D_h$ (nm)', fontsize=15)
        #ax.set(ylim=(100,250),)

    if par=='PDI':
        #list_xticks=list(df['ID'].values) + [r'$\left<\{}\right>\pm 1\sigma$'.format(par),r'$\left<\{}\right>\pm 2\sigma$'.format(par)]
        ax.set_ylabel(r'{}'.format(par), fontsize=15)
        #ax.set(ylim=(0,.3),)

    if par=='zeta':
        #list_xticks=list(df['ID'].values) + [r'$\left<{}\right>\pm 1\sigma$'.format(par[0]),r'$\left<{}\right>\pm 2\sigma$'.format(par[0])]
        ax.set_ylabel('Zeta potential (mV)', fontsize=15)
        #ax.set(ylim=(-60,-20),)

    save_results=True
    if save_results:
        store_results(os.path.join("results", "long_term_stability"), f"long_term_{par}",
                      (df, df2, df_pars), ("Data", "Table", "pars"), saveplot=True)
    plt.show()

    return df_pars

for par in ['D', 'PDI', 'zeta']:
    f_long_term_model_fit_simple(df=df_long_term, par=par, verbose=True, )

# Report Ch. 6.2: Calculation of combined uncertainty

The combined uncertainty $u_c$ was calculated according to Equation (xxx), using the numerical values summarized in Table xxx. This equation is a combination of the standard uncertainty due to characterization, the contribution from variation between bottles, and the contribution from long-term stability. Furthermore, the certified value xCRM can be assigned as ychar since no between-unit variation or stability effects need to be regarded (Equation xxx).

\begin{equation}
u_c^2=  u_{\textrm{hom}}^2 + u_{\textrm{char}}^2 + u_{\textrm{lts}}^2
\end{equation}
where the $u_i$ are the different uncertainty contributions.
The expanded uncertainty is 
\begin{equation}
U = k \times u_c
\end{equation}
with an expansion factor of $k=2$. 

### Standard uncertainty due to (in)homogeneity study $u_{\textrm{hom}}$

\begin{equation}
u_{\textrm{hom}}= \sqrt{s_r^2+ \max{(s_{bb},u'_{bb}})}
\end{equation}
The standard deviation of repeatability within bottles is

\begin{equation}
s_r=\sqrt{M_{within}}
\end{equation}

The standard uncertainty due to between-bottle variation is
\begin{equation}
s_{bb}=\sqrt{\max ({  \frac{M_{between}-M_{within}}{n} },0 ) }
\end{equation}

The maximum between-bottle variation that could be masked by within-bottle variation

\begin{equation}
u_{bb}=\sqrt{\frac{M_{within}}{n}}\times \sqrt[4]{ \frac{2}{N(n-1)} }
\end{equation}

In [ ]:
d_ANOVA={}
for item in ['D', 'PDI', 'zeta']:
    d_ANOVA[item]= pd.read_excel(os.path.join('results','homogeneity','ANOVA_'+item+'_table.xlsx'), index_col=0)
    display(d_ANOVA[item])

In [ ]:
def f_calc_uncertainty(df_ANOVA):
    """ Calculation of the uncertainty of a homogeneity study

    """
    # Overall Mean
    y_hom= df_ANOVA['Overall Mean'][0]
    print(f'y_hom = {y_hom}')
    
    # standard deviation within bottles
    s_r=df_ANOVA['Standard deviation'][1]
    print(f's_r = {s_r:2.3f}')
    
    n=3  # number of days
    N=20 # number of bottles
    M_between=df_ANOVA['Mean square'][0]
    M_within=df_ANOVA['Mean square'][1]
    
    y_brackets=(M_between-M_within)/n
    s_bb= np.sqrt( np.max([y_brackets, 0.]))
    print(f's_bb = {s_bb:2.3f}')
    
    u_bb = np.sqrt(M_within/n * (2/(N*(n-1)))**(1/4))
    print(f'u_bb = {u_bb:2.3f}')
    
    u_hom = np.sqrt(s_r**2 + np.max([s_bb,u_bb])**2)
    print(f'u_hom = {u_hom:2.3f}\n')

    return y_hom, u_hom

df_pars=pd.DataFrame()

l_pars, l_x, l_ux = [],[],[]
for item in ['D', 'PDI', 'zeta']:
    l_pars.append(item)
    print(item)
    x_CRM, u_hom =f_calc_uncertainty(d_ANOVA[item])
    l_x.append(x_CRM)
    l_ux.append(u_hom)
df_pars['parameters']= l_pars 
df_pars['x_CRM']=l_x
df_pars['u_hom']=l_ux

df_pars_hom=df_pars.copy()
df_pars_hom

In [ ]:
np.sqrt( 2.61**2 +5.15**2)

## Standard uncertainty due to characterization $u_{char}$

The hydrodynamic diameter of BAM-N010 was determined with dynamic light scattering in aqueous dispersions with three different devices.
1. ZetaSizer Nano ZS (Malvern Panalytical, follows ISO standard 22412:2017)
2. Litesizer 500 (Anton Paar AG, follows ISO standard 22412:2017)
3. ALV 7004 (ALV Langen)

The measurements of the dispersions were carried out directly after opening the vials with a volume of 1 ml without further treatment at a temperature of 25°C. Ten randomly selected bottles were measured three times each.

The assigned value $y_{char}$ for the $D_h$, PDI, and zeta potential was calculated from the average values $y_i$ obtained from the
measurements of each device and the number of devices $p = 3$ as 
\begin{equation}
y_{char}=\frac{\sum y_i}{p}.
\end{equation}

The $u_{\textrm{char}}$ is then calculated according to equation (A.4) in ISO Guide 35:2017 as
\begin{equation}
u_{char}= \frac{1}{\sqrt{p}} \sqrt{ \frac{ \sum{ (y_i-y_{char})^2} }{p-1}}.
\end{equation}

In [ ]:
print('ZetaSizer Nano ZS')
display(df_long_term.loc[0:0])

print('Litesizer 500')
display(df_long_term.loc[13:13])

print('ALV 7004')
display(df_long_term.loc[15:15])

l_devices=['ZetaSizer Nano ZS', 'Litesizer 500', 'ALV 7004']

In [ ]:
df_char=pd.concat([df_long_term.loc[0:0], df_long_term.loc[13:13], df_long_term.loc[15:15]])[['D','uD','PDI','uPDI','zeta','uzeta']]
df_char['device']=l_devices

def f_show_values(par):
    ''' Display the parameter values for the different instruments '''
    df=df_char[['device',par,'u'+par]].T
    
    # Set the first row as the header
    df.columns = df.iloc[0]
    # Set the first row as column names
    df = df[1:]
    # Remove the first row from the data
    display(df)
    return


f_show_values('D')

f_show_values('PDI')

f_show_values('zeta')

In [ ]:
def f_mean_std(df, par, digits=3):
    ''' ISO Guide 35:2017
    A.2.5 Assigned uncertainty 
    A.2.5.3 Evaluation without the laboratories' uncertainty
    Note: This is smaller than the standard deviation of the mean values
    '''
    p=len(df)
    y=df[par].sum()/p
    
    l_sum=[]
    for item in df[par]:
        l_sum.append(  (item-y)**2)
    
    # standard deviation of the p data set mean values
    s=np.sqrt(np.sum(l_sum)/(p-1))
    u_char=s/np.sqrt(p)
    u_rel=np.abs(u_char/y*100)

    df_res=pd.DataFrame()
    df_res['parameter']=[par]
    df_res['y_char']= [round(y,digits)]
    
    df_res['u_char']= [round(u_char,digits)] 
    df_res['u_rel']= [round(u_rel,digits)]

    return df_res


l_dfs=[]
d_ychar={}
d_ychar['D']=f_mean_std(df_char, 'D', digits=1)
display(d_ychar['D'])

d_ychar['PDI']=f_mean_std(df_char, 'PDI', digits=3)
display(d_ychar['PDI'])

d_ychar['zeta']=f_mean_std(df_char, 'zeta', digits=1)
display(d_ychar['zeta'])

for item in ['D', 'PDI', 'zeta']:
    l_dfs.append(d_ychar[item])

df_pars_char=pd.concat(l_dfs)
df_pars_char.reset_index(inplace=True, drop=True)
df_pars_char

## Standard uncertainty due to long-term instability $u_{lts}$

The standard uncertainty of the long-term stability was calculated as
\begin{equation}
u_{lts}= s(b_1) (t_m+t_{cert}),
\end{equation}
where $s(b_1)$ is the standard uncertainty of the slope, $t_m$ is the time interval between value assignment (40 months) and the beginning of the stability monitoring.
The $t_{cert}$ is the period of validity issued (12 months).

In [ ]:
df_lts_D=pd.read_excel(os.path.join('results','long_term_stability','long_term_D_pars.xlsx'), index_col=0)
display(df_lts_D)

df_lts_PDI=pd.read_excel(os.path.join('results','long_term_stability','long_term_PDI_pars.xlsx'), index_col=0)
display(df_lts_PDI)

df_lts_zeta=pd.read_excel(os.path.join('results','long_term_stability','long_term_zeta_pars.xlsx'), index_col=0)
display(df_lts_zeta)

In [ ]:
# time of stability study
t_m=12.
# time of validity
t_cert=12.

l_dfs=[]
for item in ['D', 'PDI', 'zeta']:
    df=pd.read_excel(os.path.join('results','long_term_stability','long_term_'+item+'_pars.xlsx'), index_col=0)
    l_dfs.append(df)
df_pars_lts=pd.concat(l_dfs)
df_pars_lts.reset_index(inplace=True, drop=True)
df_pars_lts['u_lts']= df_pars_lts['s_b1']*(t_m+t_cert)
display(df_pars_lts)

#Combined uncertainty $u_c^2$

\begin{equation}
u_c^2= u_{\text{hom}}^2 + u_{\text{char}}^2 + u_{\text{lts}}^2
\end{equation}
where the $u_i$ are the different uncertainty contributions.
The expanded uncertainty is 
\begin{equation}
U = k \times u_c
\end{equation}
with an expansion factor of $k=2$. 

In [ ]:
df_pars=pd.concat([df_pars_hom, df_pars_char['u_char'], df_pars_lts['u_lts'] ], axis=1)
df_pars['u_c']=  np.sqrt(df_pars['u_hom']**2 + df_pars['u_char']**2 + df_pars['u_lts']**2)
df_pars['U']=2*df_pars['u_c']
display(df_pars)


u_c=np.sqrt(sum([ item**2 for item in [4.6, 5.8, 3.7]]))
U = 2*u_c
print(f'u_c = {u_c}, U = {U}')